# AI-Powered E-Commerce Customer Intelligence

## Notebook 02 — Data Preparation

### Objective

Notebook ini mempersiapkan raw dataset agar dapat digunakan untuk
analisis bisnis dan machine learning.

Tahapan utama:

1. Load raw dataset
2. Validate data types
3. Convert date columns
4. Validate categorical values
5. Define completed transactions
6. Prepare analytical transaction data
7. Prepare customer-level dataset
8. Validate the processed datasets

### Important Principle

Data preparation dilakukan berdasarkan business rules.

Kita tidak menghapus data hanya karena data tersebut tidak termasuk
dalam transaksi completed.

Transaction dengan status:

- completed
- refunded
- cancelled
- pending

tetap dipertahankan pada raw dataset.

Untuk analisis revenue, hanya transaksi `completed` yang akan
digunakan sebagai dasar revenue metric.

## 1. Environment & Imports

### Tujuan

Memuat library yang diperlukan untuk proses data preparation.

### Libraries

- Pandas: data manipulation
- NumPy: numerical computation
- Pathlib: filesystem path management

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Load Raw Dataset

### Tujuan

Memuat dataset raw dari folder:

`data/raw/`

Dataset tidak dimodifikasi secara langsung.

Seluruh transformation akan menghasilkan dataset baru sehingga raw data
tetap dapat digunakan sebagai source of truth.

In [2]:
DATA_DIR = Path("../data/raw")

customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
reviews = pd.read_csv(DATA_DIR / "reviews.csv")

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Transactions:", transactions.shape)
print("Sessions:", sessions.shape)
print("Reviews:", reviews.shape)

Customers: (10000, 10)
Products: (1000, 11)
Transactions: (120000, 11)
Sessions: (80000, 10)
Reviews: (25000, 8)


## 3. Date Type Conversion

### Tujuan

Mengubah seluruh kolom tanggal menjadi Pandas datetime.

### Date columns

- `customers.signup_date`
- `transactions.transaction_date`
- `sessions.session_date`
- `reviews.review_date`

### Alasan

Datetime diperlukan untuk:

- time-based aggregation
- customer tenure
- monthly sales
- cohort analysis
- churn prediction
- sales forecasting

### Output

Seluruh date column harus memiliki dtype `datetime64`.

In [3]:
date_columns = {
    "customers": ["signup_date"],
    "transactions": ["transaction_date"],
    "sessions": ["session_date"],
    "reviews": ["review_date"],
}

for table_name, columns in date_columns.items():
    df = {
        "customers": customers,
        "transactions": transactions,
        "sessions": sessions,
        "reviews": reviews,
    }[table_name]

    for column in columns:
        df[column] = pd.to_datetime(df[column])

print(customers["signup_date"].dtype)
print(transactions["transaction_date"].dtype)
print(sessions["session_date"].dtype)
print(reviews["review_date"].dtype)

datetime64[us]
datetime64[us]
datetime64[us]
datetime64[us]


## 4. Categorical Value Validation

### Tujuan

Memeriksa nilai kategori yang tersedia sebelum menentukan business rules.

### Variables

Customers:

- gender
- country
- segment

Transactions:

- status
- payment_method

Sessions:

- device
- channel

### Alasan

Kita perlu mengetahui nilai aktual dataset sebelum melakukan filtering
atau transformation.

In [4]:
categorical_columns = {
    "customers.gender": customers["gender"],
    "customers.segment": customers["segment"],
    "transactions.status": transactions["status"],
    "transactions.payment_method": transactions["payment_method"],
    "sessions.device": sessions["device"],
    "sessions.channel": sessions["channel"],
}

for name, series in categorical_columns.items():
    print(f"\n{name}")
    print(series.value_counts(dropna=False))


customers.gender
gender
F                    4563
M                    4466
Prefer not to say     495
Non-binary            476
Name: count, dtype: int64

customers.segment
segment
Regular               3500
Budget Shopper        3043
Premium               1982
VIP                    974
Occasional Visitor     501
Name: count, dtype: int64

transactions.status
status
completed    68700
refunded     17139
cancelled    17092
pending      17069
Name: count, dtype: int64

transactions.payment_method
payment_method
credit_card      41769
debit_card       24163
paypal           21721
apple_pay        14350
google_pay       11977
bank_transfer     6020
Name: count, dtype: int64

sessions.device
device
mobile     40010
desktop    28053
tablet     11937
Name: count, dtype: int64

sessions.channel
channel
organic        19974
paid_search    15915
social         12071
direct         12067
email          11854
referral        8119
Name: count, dtype: int64


## 5. Completed Transaction Definition

### Business Rule

Untuk sales/revenue analysis, transaksi yang digunakan sebagai
completed sales adalah transaksi dengan:

`status == "completed"`

### Rationale

Raw transaction table berisi beberapa status:

- completed
- refunded
- cancelled
- pending

Tidak semua status merepresentasikan revenue yang sudah terealisasi.

### Important

Raw transactions tetap disimpan lengkap.

Kita hanya membuat filtered analytical dataset untuk kebutuhan
sales analysis.

In [5]:
completed_transactions = transactions[
    transactions["status"].eq("completed")
].copy()

print("Raw transactions:", len(transactions))
print("Completed transactions:", len(completed_transactions))

Raw transactions: 120000
Completed transactions: 68700


## 6. Transaction Amount Validation

### Tujuan

Memeriksa hubungan antara:

- quantity
- unit_price
- total_amount

### Expected Relationship

Secara umum:

`quantity × unit_price ≈ total_amount`

Namun dataset juga memiliki:

`discount_applied`

sehingga perbedaan dapat terjadi.

Kita tidak langsung mengubah `total_amount`.

Tujuan tahap ini adalah memahami bagaimana nilai transaksi
dibentuk oleh dataset.

In [6]:
completed_transactions["calculated_amount"] = (
    completed_transactions["quantity"]
    * completed_transactions["unit_price"]
)

completed_transactions["amount_difference"] = (
    completed_transactions["total_amount"]
    - completed_transactions["calculated_amount"]
)

completed_transactions[
    [
        "quantity",
        "unit_price",
        "discount_applied",
        "total_amount",
        "calculated_amount",
        "amount_difference",
    ]
].head(10)

,quantity,unit_price,discount_applied,total_amount,calculated_amount,amount_difference
2,1,67.75,0,67.75,67.75,0.0
4,1,11.97,50,11.97,11.97,0.0
5,1,27.11,20,27.11,27.11,0.0
6,1,67.14,0,67.14,67.14,0.0
8,1,24.20,0,24.20,24.20,0.0
10,1,29.54,0,29.54,29.54,0.0
11,2,33.25,0,66.50,66.50,0.0
12,1,28.66,15,28.66,28.66,0.0
13,1,4.88,0,4.88,4.88,0.0
15,1,11.60,10,11.60,11.60,0.0


## 7. Transaction-Level Analytical Dataset

### Tujuan

Menggabungkan completed transactions dengan product information.

### Join

`transactions.product_id → products.product_id`

### Information yang ditambahkan

- product_name
- category
- brand
- price
- avg_rating
- discount_pct

### Alasan

Dataset transaksi sendiri hanya memberikan `product_id`.

Untuk business analysis, kita membutuhkan informasi produk seperti
category dan brand.

In [7]:
transaction_analysis = completed_transactions.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_product"),
)

print("Transaction analysis shape:", transaction_analysis.shape)

transaction_analysis.head()

Transaction analysis shape: (68700, 23)


,transaction_id,customer_id,product_id,transaction_date,quantity,unit_price,total_amount,discount_applied,status,payment_method,...,product_name,category,brand,price,avg_rating,num_ratings,stock_quantity,discount_pct,is_featured,weight_kg
0,T117783,C05855,P0065,2023-01-01 00:45:53,1,67.75,67.75,0,completed,apple_pay,...,EcoLiving Home & Garden #65,Home & Garden,EcoLiving,67.75,3.4,153,299,0,0,0.76
1,T092643,C00760,P0881,2023-01-01 01:06:45,1,11.97,11.97,50,completed,paypal,...,ClassicCo Automotive #881,Automotive,ClassicCo,23.94,4.1,420,871,50,0,0.72
2,T088042,C02979,P0269,2023-01-01 01:13:19,1,27.11,27.11,20,completed,credit_card,...,GlowUp Health #269,Health,GlowUp,33.89,3.7,308,163,20,0,4.09
3,T110118,C03752,P0683,2023-01-01 01:17:34,1,67.14,67.14,0,completed,debit_card,...,SoundWave Automotive #683,Automotive,SoundWave,67.14,3.8,174,661,0,0,1.03
4,T104403,C00519,P0638,2023-01-01 01:26:34,1,24.20,24.20,0,completed,paypal,...,PetLife Office Supplies #638,Office Supplies,PetLife,24.20,3.3,214,983,0,0,0.34


## 8. Customer Analytical Dataset

### Tujuan

Membuat dataset pada level customer.

Satu row akan merepresentasikan satu customer.

### Information

Customer profile akan digabungkan dengan aggregated transaction
information.

### Planned Features

- total_orders
- total_quantity
- total_revenue
- average_order_value
- first_purchase_date
- last_purchase_date

Feature tambahan untuk RFM dan churn prediction akan dibuat pada
tahap feature engineering.

### Important

Aggregation dilakukan menggunakan completed transactions agar
revenue metric tidak tercampur dengan cancelled, pending, atau
refunded transactions.

In [8]:
customer_transaction_summary = (
    completed_transactions
    .groupby("customer_id")
    .agg(
        total_orders=("transaction_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_revenue=("total_amount", "sum"),
        average_order_value=("total_amount", "mean"),
        first_purchase_date=("transaction_date", "min"),
        last_purchase_date=("transaction_date", "max"),
    )
    .reset_index()
)

customer_transaction_summary.head()

,customer_id,total_orders,total_quantity,total_revenue,average_order_value,first_purchase_date,last_purchase_date
0,C00000,18,23,706.65,39.258333,2023-05-12 00:04:32,2024-12-26 20:35:40
1,C00001,5,9,244.64,48.928000,2023-03-31 10:58:45,2024-12-23 13:01:19
2,C00002,16,17,1104.66,69.041250,2023-01-21 05:15:47,2024-07-13 19:03:43
3,C00003,15,23,695.06,46.337333,2023-01-07 16:47:15,2024-12-23 09:32:31
4,C00004,2,2,148.32,74.160000,2023-03-26 21:10:06,2024-04-17 04:55:28


## 9. Customer Profile + Transaction Summary

### Tujuan

Menggabungkan customer profile dengan historical purchase behavior.

### Join

`customers.customer_id → customer_transaction_summary.customer_id`

### Join Type

`left`

### Alasan

Seluruh customer harus tetap dipertahankan, termasuk customer yang
belum memiliki completed transaction.

Customer tanpa transaksi akan memiliki nilai transaction features
yang kosong dan akan ditangani pada tahap berikutnya berdasarkan
business context.

In [9]:
customer_analysis = customers.merge(
    customer_transaction_summary,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

print("Customer analysis shape:", customer_analysis.shape)

customer_analysis.head()

Customer analysis shape: (10000, 16)


,customer_id,signup_date,age,gender,country,segment,is_churned,lifetime_value,email_opt_in,has_app,total_orders,total_quantity,total_revenue,average_order_value,first_purchase_date,last_purchase_date
0,C00000,2020-02-13,28,M,BR,Premium,0,1595.27,0,0,18.0,23.0,706.65,39.258333,2023-05-12 00:04:32,2024-12-26 20:35:40
1,C00001,2021-10-01,22,Prefer not to say,FR,Regular,0,1160.61,1,0,5.0,9.0,244.64,48.928000,2023-03-31 10:58:45,2024-12-23 13:01:19
2,C00002,2022-06-26,30,F,US,VIP,0,3093.32,1,1,16.0,17.0,1104.66,69.041250,2023-01-21 05:15:47,2024-07-13 19:03:43
3,C00003,2021-12-21,48,F,US,Premium,1,2131.08,1,0,15.0,23.0,695.06,46.337333,2023-01-07 16:47:15,2024-12-23 09:32:31
4,C00004,2021-09-16,37,M,CA,Budget Shopper,0,583.39,0,1,2.0,2.0,148.32,74.160000,2023-03-26 21:10:06,2024-04-17 04:55:28


## 10. Processed Dataset Validation

### Tujuan

Memastikan hasil transformation tidak mengubah struktur dasar data
secara tidak sengaja.

### Checks

- Customer count
- Transaction count
- Duplicate customer IDs
- Duplicate transaction IDs
- Missing values introduced by joins

In [10]:
validation_summary = {
    "original_customers": len(customers),
    "processed_customers": len(customer_analysis),
    "original_transactions": len(transactions),
    "completed_transactions": len(completed_transactions),
    "duplicate_customer_ids": customer_analysis["customer_id"].duplicated().sum(),
    "duplicate_transaction_ids": transaction_analysis["transaction_id"].duplicated().sum(),
}

pd.Series(validation_summary)

original_customers            10000
processed_customers           10000
original_transactions        120000
completed_transactions        68700
duplicate_customer_ids            0
duplicate_transaction_ids         0
dtype: int64

## 11. Missing Values After Transformation

### Tujuan

Membedakan missing values yang berasal dari masalah data dengan
missing values yang memang merupakan konsekuensi dari `left join`.

Contoh:

Customer yang tidak pernah melakukan completed transaction akan
memiliki:

- total_orders = NaN
- total_revenue = NaN
- first_purchase_date = NaN
- last_purchase_date = NaN

Kondisi tersebut tidak otomatis berarti data rusak.

Customer tersebut dapat merupakan legitimate customer.

In [11]:
customer_analysis.isna().sum().sort_values(ascending=False)

total_revenue          270
average_order_value    270
last_purchase_date     270
first_purchase_date    270
total_orders           270
total_quantity         270
customer_id              0
signup_date              0
lifetime_value           0
is_churned               0
segment                  0
country                  0
gender                   0
age                      0
email_opt_in             0
has_app                  0
dtype: int64

## 12. Save Processed Datasets

### Tujuan

Menyimpan hasil data preparation sehingga notebook berikutnya tidak
perlu selalu melakukan seluruh transformation dari raw dataset.

### Output

- `completed_transactions.csv`
- `transaction_analysis.csv`
- `customer_analysis.csv`

### Data Lineage

Raw Data
→ Data Preparation
→ Processed Data
→ EDA / Feature Engineering / Machine Learning

In [12]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

completed_transactions.to_csv(
    PROCESSED_DIR / "completed_transactions.csv",
    index=False,
)

transaction_analysis.to_csv(
    PROCESSED_DIR / "transaction_analysis.csv",
    index=False,
)

customer_analysis.to_csv(
    PROCESSED_DIR / "customer_analysis.csv",
    index=False,
)

print(f"Processed datasets saved to: {PROCESSED_DIR.resolve()}")

Processed datasets saved to: D:\Project\ai-business-intelligence\data\processed


## 13. Data Preparation Summary

### Completed

- Raw datasets loaded.
- Date columns converted to datetime.
- Categorical values inspected.
- Completed transaction dataset created.
- Transaction amount relationship validated.
- Transaction and product information joined.
- Customer-level transaction aggregation created.
- Customer profile and transaction behavior combined.
- Processed datasets validated.
- Processed datasets exported.

### Output Datasets

`data/processed/completed_transactions.csv`

`data/processed/transaction_analysis.csv`

`data/processed/customer_analysis.csv`

### Next Step

The next notebook will focus on:

**Exploratory Data Analysis (EDA)**

The EDA will answer the business questions using the prepared data
before feature engineering and machine learning.